In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

# ---------------------------------------------------------
# 1. Load final dataset ensuring text format for ID codes
# ---------------------------------------------------------

BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)

df = pd.read_csv(Path(BASE_DIR, "panel_final_ofrecen11.csv"),
    dtype={
        "SEDE_CODIGO": str,
        "COD_MPIO_DANE": str,
        "COD_DPTO_DANE": str,
    },
)
total_rows = len(df)

# ---------------------------------------------------------
# 2. Calculate Data Quality Dimensions
# ---------------------------------------------------------

# Completeness: % of non-null cells across key model variables
key_cols = [
    "SEDE_CODIGO",
    "ANIO_OBJETIVO",
    "ANIO_REFERENCIA",
    "SPLIT",
    "COD_MPIO_DANE",
    "DESERCION_LAG2",
    "REPITENCIA_LAG2",
    "MATRICULA_TOTAL",
    "MATRICULA_PCT_CAMBIO",
    "PROP_DESPLAZADOS",
    "PROP_LIMITACION",
    "PROP_ETNIA",
    "PROP_FLEXIBLE",
    "ICFES_PROM_LECTURA",
    "ICFES_PROM_MATEMATICAS",
    "ICFES_PROM_INSE",
    "TASA_DESERCION_MPIO",
]
total_cells = df[key_cols].size
null_cells = df[key_cols].isnull().sum().sum()
completeness = ((total_cells - null_cells) / total_cells) * 100

# Uniqueness: % of records without duplicates on the composite key (SEDE_CODIGO, ANIO_OBJETIVO)
duplicates = df.duplicated(subset=["SEDE_CODIGO", "ANIO_OBJETIVO"]).sum()
uniqueness = ((total_rows - duplicates) / total_rows) * 100

# Consistency: Temporal coherence (ANIO_REFERENCIA = t - 2) and logical proportions within [0, 1]
consistent_year = (df["ANIO_REFERENCIA"] == df["ANIO_OBJETIVO"] - 2).mean() * 100
prop_cols = [
    "PROP_DESPLAZADOS",
    "PROP_LIMITACION",
    "PROP_ETNIA",
    "PROP_FLEXIBLE",
]
prop_valid = sum(
    ((df[c].dropna() >= 0) & (df[c].dropna() <= 1)).sum() for c in prop_cols
)
prop_total = sum(len(df[c].dropna()) for c in prop_cols)
consistency_prop = (prop_valid / prop_total) * 100 if prop_total > 0 else 100
consistency = (consistent_year * 0.5) + (consistency_prop * 0.5)

# Validity: Format standards (12-digit school code) and ICFES scores in range [0, 100]
val_sede = (df["SEDE_CODIGO"].astype(str).str.len() == 12).mean() * 100
icfes_cols = [
    "ICFES_PROM_LECTURA",
    "ICFES_PROM_MATEMATICAS",
    "ICFES_PROM_CIENCIAS",
    "ICFES_PROM_SOCIALES",
    "ICFES_PROM_INGLES",
]
icfes_valid = sum(
    ((df[c].dropna() >= 0) & (df[c].dropna() <= 100)).sum() for c in icfes_cols
)
icfes_total = sum(len(df[c].dropna()) for c in icfes_cols)
val_icfes = (icfes_valid / icfes_total) * 100 if icfes_total > 0 else 100
validity = (val_sede * 0.5) + (val_icfes * 0.5)

# Accuracy: Consistency in base operational metrics (total enrollment >= 0)
mat_valid = (df["MATRICULA_TOTAL"].dropna() >= 0).mean() * 100
accuracy = mat_valid

# Total Quality: Weighted average of evaluated quality dimensions
dimensions_dict = {
    "Completeness": completeness,
    "Uniqueness": uniqueness,
    "Consistency": consistency,
    "Validity": validity,
    "Accuracy": accuracy,
}
total_quality = np.mean(list(dimensions_dict.values()))
dimensions_dict["Total Quality"] = total_quality

# ---------------------------------------------------------
# 3. Consolidate and Print Results Table
# ---------------------------------------------------------
df_results = pd.DataFrame(
    list(dimensions_dict.items()), columns=["Dimension", "% Compliance"]
)
print("=== DATA QUALITY REPORT TABLE ===")
print(df_results.to_string(index=False))

# ---------------------------------------------------------
# 4. Generate and Save Horizontal Bar Chart Visualization
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
ax = sns.barplot(
    x="% Compliance", y="Dimension", data=df_results, palette="crest"
)
plt.title(
    "Data Quality Evaluation - Modeling Panel (ofrecen11)",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
plt.xlabel("% Compliance", fontsize=12)
plt.ylabel("Quality Dimension", fontsize=12)
plt.xlim(0, 105)

# Annotate percentage values on bars
for p in ax.patches:
    width = p.get_width()
    ax.annotate(
        f"{width:.2f}%",
        (width + 1.5, p.get_y() + p.get_height() / 2.0),
        ha="left",
        va="center",
        fontsize=11,
        color="black",
        fontweight="semibold",
    )

plt.tight_layout()
plt.savefig("data_quality_panel.png", dpi=300)
plt.show()